# **Day 16: Robust Model Evaluation**

### Without Shuffling:-

In [1]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_digits
import numpy as np

# Load Data
digits = load_digits()
X, y = digits.data, digits.target

# Model
model = RandomForestClassifier(random_state=42)

# KFold WITHOUT shuffle
kf_no_shuffle = KFold(n_splits=10, shuffle=False)

scores_no_shuffle = cross_val_score(model, X, y, cv=kf_no_shuffle)

print("Without Shuffle:")
print("Scores:", scores_no_shuffle)
print("Mean:", np.mean(scores_no_shuffle))
print("Std Dev:", np.std(scores_no_shuffle))

Without Shuffle:
Scores: [0.89444444 0.98333333 0.96111111 0.95555556 0.95       0.97777778
 0.96111111 0.96089385 0.94413408 0.9273743 ]
Mean: 0.9515735567970204
Std Dev: 0.02426347070116984


### With Shuffling:-

In [2]:
# KFold WITH shuffle
kf_shuffle = KFold(n_splits=10, shuffle=True, random_state=42)

scores_shuffle = cross_val_score(model, X, y, cv=kf_shuffle)

print("\nWith Shuffle:")
print("Scores:", scores_shuffle)
print("Mean:", np.mean(scores_shuffle))
print("Std Dev:", np.std(scores_shuffle))


With Shuffle:
Scores: [0.97777778 0.97222222 0.98888889 0.98888889 0.96111111 0.96111111
 0.98888889 0.97765363 0.98882682 0.98882682]
Mean: 0.9794196151458723
Std Dev: 0.010835182102050938



##### Observations:

* Without shuffling, the model achieved ~95.16% accuracy, while with shuffling it improved to ~97.94%.
* The standard deviation reduced significantly from 0.0243 to 0.0108, indicating much better stability.
* Without shuffle, the scores varied more (0.89 → 0.98), showing inconsistent performance across folds.
* With shuffle, the scores are tightly clustered (~0.96 → 0.99), showing consistent behavior.
* Data without shuffling may contain ordering bias, causing some folds to be easier or harder.
* Shuffling ensures uniform data distribution across all folds.
* Lower standard deviation after shuffling proves the model is more reliable and less sensitive to data splits.
* The experiment confirms that K-Fold with shuffle=True provides a more robust and unbiased evaluation.

### Stratified K Fold Cross Validation

Stratified K-Fold Cross Validation is a technique used for evaluating a model. It is particularly useful for
classification problems in which the class labels are not evenly distributed i.e data is imbalanced.
It is a enhanced version of K-Fold Cross Validation. Key difference is that it uses stratification which
allows original distribution of each class to be maintained across each fold.

For example, if your original dataset had 80% Class 0 and 20% Class 1 your folds would reflect the
same proportion of classes in your data. This creates improved and more reliable accuracy metrics.


#### Problem with Random Splitting

Random splitting techniques like train_test_split() or regular K-Fold can create problem if they produce imbalanced class proportions in the training and test sets. For example imagine a binary classification dataset with 100 samples where:

* 80 samples are Class 0
* 20 samples are Class 1
Using random sampling in an 80:20 split then all 80 Class 0 in the training set and all 20 Class 1 in the test set. In this case model will never learn to classify Class 1 and would give misleading accuracy.

Now, let’s use stratified sampling on same dataset:

1. Training Set (80 samples):
* 64 from Class 0 (80% of 80)
* 16 from Class 1 (80% of 20)
2. Test Set (20 samples):

* 16 from Class 0 (20% of 80)
* 4 from Class 1 (20% of 20)
This ensures that both training and test sets provide an accurate representation of the full dataset's class proportions and better generalization in the evaluation set.

In real-world classification tasks distribution of observations per class is often imbalanced like in a medical dataset it could be the case that 90% of patients are healthy (Class 0) and 10% have a disease (Class 1). If we randomly split this data there may be some training/test sets that have very few sample or even no samples for the minority class that where Stratified K Fold Cross Validation becomes important.

#### Importing Required Libraries:

In [3]:
from statistics import mean, stdev
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold
from sklearn import linear_model
from sklearn import datasets

#### Loading Dataset and Extracting Features:
Here we will be using breast cancer dataset available in scikit learn.

* x = cancer.data: feature/input values
* y = cancer.target: output/class labels (0 or 1)

In [4]:
cancer = datasets.load_breast_cancer()

x = cancer.data
y = cancer.target

#### Feature Scaling (Normalization):
* MinMaxScaler(): scales features to a range between 0 and 1
* fit_transform(x): fits scaler on data and applies transformation

In [5]:
scaler = preprocessing.MinMaxScaler()
x_scaled = scaler.fit_transform(x)


#### Model and K-Fold Object Setup:

Here we will be using logistic regression model.

* StratifiedKFold(...): sets up 10-fold stratified cross-validation.
* lst_accu_stratified: empty list to store accuracy scores.

In [6]:
lr = linear_model.LogisticRegression()

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
lst_accu_stratified = []


####  Applying Stratified K-Fold and Training Model:

* skf.split(x, y): splits dataset into stratified train-test indices.
* x_train_fold, x_test_fold: features for training and testing.
* y_train_fold, y_test_fold: labels for training and testing.

In [7]:
for train_index, test_index in skf.split(x, y):
	x_train_fold, x_test_fold = x_scaled[train_index], x_scaled[test_index]
	y_train_fold, y_test_fold = y[train_index], y[test_index]
	lr.fit(x_train_fold, y_train_fold)
	lst_accu_stratified.append(lr.score(x_test_fold, y_test_fold))

#### Accuracy Results:

In [8]:
print('List of possible accuracy:', lst_accu_stratified)
print('\nMaximum Accuracy That can be obtained from this model is:',
	max(lst_accu_stratified)*100, '%')
print('\nMinimum Accuracy:',
	min(lst_accu_stratified)*100, '%')
print('\nOverall Accuracy:',
	mean(lst_accu_stratified)*100, '%')
print('\nStandard Deviation is:', stdev(lst_accu_stratified))

List of possible accuracy: [0.9298245614035088, 0.9649122807017544, 0.9824561403508771, 1.0, 0.9649122807017544, 0.9649122807017544, 0.9824561403508771, 0.9473684210526315, 0.9473684210526315, 0.9821428571428571]

Maximum Accuracy That can be obtained from this model is: 100.0 %

Minimum Accuracy: 92.98245614035088 %

Overall Accuracy: 96.66353383458647 %

Standard Deviation is: 0.02097789213195869


#### Observation:
Here we can see that we got a overall accuracy of 96.6% and standard deviation of 0.02 which means our model is working fine.

By using Stratified K-Fold Cross Validation we can ensure that our machine learning model is evaluated fairly and consistently leading to more accurate predictions and better real-world performance.

# **Reflect:**


**In a Markdown cell, answer: "On MeetMux, we might have 10,000 users
from Bangalore and only 100 from Delhi. If we don't use Shuffling in our K-Fold, how
could that ruin our model's ability to predict Delhi users' preferences?"**


## Impact of Not Using Shuffling in K-Fold

If our MeetMux dataset contains 10,000 users from Bangalore and only 100 users from Delhi, the data is **highly imbalanced** and likely ordered by location.

Without shuffling, K-Fold Cross Validation may create folds where:

* Some folds contain **mostly Bangalore users**
* Some folds contain **very few or mostly Delhi users**

This creates multiple problems:

* The model is trained primarily on **Bangalore data**, so it learns patterns specific to Bangalore users.
* Delhi users are **underrepresented**, so their preferences are not properly learned.
* When a test fold contains more Delhi users, the model may perform **poorly**, leading to inconsistent accuracy.
* This results in **biased predictions**, where the model performs well for Bangalore users but fails for Delhi users.
* The evaluation becomes **unreliable**, with high variation across folds.

By enabling shuffling, each fold gets a **balanced mix of Bangalore and Delhi users**, ensuring:

* Better learning of all user groups
* Fair evaluation
* Improved generalization

**Conclusion:** Without shuffling, the model becomes biased toward the majority class (Bangalore users) and fails to generalize to minority groups (Delhi users), making it unreliable for real-world use.


## Why Stratified K-Fold is Important for MeetMux

If MeetMux has 10,000 users from Bangalore and only 100 users from Delhi, the dataset is highly imbalanced. In such a case, using normal K-Fold without shuffling or stratification can seriously distort model evaluation.

Without stratification, some folds may contain mostly Bangalore users while others may contain very few Delhi users. This causes the model to:

* Learn Bangalore user patterns much more than Delhi users
* Fail to properly understand Delhi user preferences
* Show inconsistent performance when tested on different folds

As a result, the model may look accurate overall, but it will perform poorly for the minority group (Delhi users), which is a critical failure in real-world systems like MeetMux.

Stratified K-Fold solves this problem by ensuring that each fold maintains the same proportion of Bangalore and Delhi users as in the original dataset. This leads to:

* Fair representation of both user groups in every fold
* Better learning of minority class patterns (Delhi users)
* More reliable and unbiased evaluation of model performance
* Improved generalization in real-world scenarios

**Conclusion:** Stratified K-Fold ensures that the model does not become biased toward Bangalore users and is equally capable of predicting preferences for Delhi users, making it essential for imbalanced datasets like MeetMux.


### AI Pro-Tip (Industry Insight)

In real-world machine learning systems, we never rely on a single accuracy value because it can be misleading. Instead, we focus on **consistency across multiple validations**.


####  Why Standard Deviation Matters
Cross-validation gives multiple accuracy scores. The **Standard Deviation (σ)** tells us how much these scores vary.

**σ>0.0 ⇒ Unstable Model**


#### If Standard Deviation is High (> 0.05)

* Model performance changes significantly across different data splits
* Indicates **high sensitivity to training data**
* Model may be:

  * Overfitting certain patterns
  * Failing on unseen distributions
* Real users will experience **inconsistent predictions**

 Result: **Model is NOT production-ready**



#### If Standard Deviation is Low (< 0.05)

* Scores are consistent across folds
* Model behaves similarly on different data samples
* Indicates:

  * Stable learning
  * Good generalization
  * Reliable performance

 Result: **Model is safe for real-world deployment**



#### Industry Conclusion

In production ML systems, a model is not judged only by accuracy, but by:

* Mean performance (how good it is overall)
* Standard deviation (how stable it is)

A model with **high accuracy but high variance is considered risky**, while a slightly lower but stable model is preferred in production.


####  Final Insight

 In industry, stability matters more than peak accuracy. A consistent 94% model is far better than an unstable 98% model that fails unpredictably in real user scenarios.
